# Calculate depth to water table from the surface from ATS simulation outputs

##### The following script calculates the water table depth from ATS-simulated subsurface pressure.
##### It employs a bottom-up approach to identify the last saturated cell starting from the bottom soil layers. In other words, the pressure of the last saturated cell, indexed from the bottom (where the bottom-most subsurface cell has an index of 0), is used to calculate the water table depth.

#### Formula

##### Water_Table_depth = z_surface_coords - z_cell_coords - [Pressure_cell - Pressure_atm] / [water_density * g]

##### where:
- `z_surface_coords`, `z_cell_coords` are the z coordinates [m] of the surface and the identified last saturated subsurface cell, respectively.
- `Pressure_cell` is the corresponding cell's subsurface pressure [Pascals].


### User Inputs


In [ ]:
# Define the output model directory, whre ats_vis data (.h5) are located
model_dir = './run2-transient/Oak-0/'

# Define the Parameters AND verify with the XML
rho = 997 # density of water, kg m^-3
g = 9.80665 # gravity, m s^-2
patm = 101325 # atmopsheric pressure, Pascals

### Loading necessary pacakges

In [ ]:
import ats_xdmf as xdmf
import numpy as np
import time
import random
import pandas
import os
import h5py
import matplotlib.pyplot as plt

### function to estimate WTD based on pressure 

In [ ]:
def get_ats_wtd_pressurebased(pressure_subsurface,visfile_surface, visfile_subsurface):
    # visfile_subsurface.centroids.shape -> (n_surface, 14, 3); 14 is soil layers, bottom-top
    iz_coord = visfile_subsurface.centroids[:,:,-1]
    
    ### Find Equivalent Surface and Subsurface ID based on cell centroids
    # take some care on the rounding of coordinates, can cause errors
    surface_centroids_rounded = np.round(visfile_surface.centroids[:, :2], 4)
    subsurface_centroids_rounded = np.round(visfile_subsurface.centroids[:, 0, :2], 4)
    # [remarks] prepare to perform a pairwise compare [n_surface,1,2] with [1,n_surface,2] -> returns [n_surface, n_surface]
    surface_centroids_expanded = surface_centroids_rounded[:, np.newaxis, :]
    subsurface_centroids_expanded = subsurface_centroids_rounded[np.newaxis, :, :]
    matches = np.all(surface_centroids_expanded == subsurface_centroids_expanded, axis=-1)
    surface_indices, subsurface_indices = np.nonzero(matches)
    surface_subsurface_IDs = np.column_stack((surface_indices, subsurface_indices))
    assert surface_subsurface_IDs[:,1].shape == visfile_surface.centroids[:,1].shape, f"Shape mismatch: change the round precision in the surface/subsurface centroid coordinates"
    
    ### Estimate WTD based on pressure
    # pressure head
    ih = (pressure_subsurface - patm) / (rho * g) # dim=(time, n_surface, 14)
    mask = ih > 0
    first_false_idx = (~mask).argmax(axis=-1) # dim=(time, n_surface)
    max_len = mask.shape[-1]
    indices = np.arange(max_len)
    mask2 = indices < first_false_idx[..., np.newaxis]
    all_true_rows = (first_false_idx == 0)
    mask2[all_true_rows] = True
    sat_idx = mask2[:, :, ::-1].argmax(axis=-1)
    sat_idx = mask2.shape[-1] - 1 - sat_idx
    sat_idx[~mask2.any(axis=-1)] = 0
    # WTD elevation
    # [remark] iH_rev=part1+part2; part1 is the relative distance between water table and the first saturated subsurface cell
    iH_rev = ih[np.arange(ih.shape[0])[:, None], np.arange(ih.shape[1]), sat_idx] + iz_coord[np.arange(ih.shape[1]), sat_idx] 
    first_depth_to_centroid = visfile_surface.centroids[surface_subsurface_IDs[0][0], -1] - visfile_subsurface.centroids[surface_subsurface_IDs[0][1], -1, -1]
    # WTD (from surface)
    head_pressure_based = -(iz_coord[:,-1]+first_depth_to_centroid - iH_rev) #[added by Yi]
    wtdep_pressure_based = np.maximum(iz_coord[:,-1]+first_depth_to_centroid - iH_rev, 0)
    pwdep_pressure_based = - np.minimum(iz_coord[:,-1]+first_depth_to_centroid - iH_rev, 0)
    assert pressure_subsurface[:,:,1].shape == wtdep_pressure_based.shape, f"Shape mismatch: Error in WTD calculation"
    
    ### Re-arrange WTD in accordance to surface IDs
    # As pressure obtained from subsurface does not follow surface ID, re-arrange this:
    head_rearranged  = head_pressure_based[:, subsurface_indices]
    wtdep_rearranged = wtdep_pressure_based[:, subsurface_indices]
    pwdep_rearranged = pwdep_pressure_based[:, subsurface_indices]
    
    return surface_subsurface_IDs, head_rearranged, wtdep_rearranged, pwdep_rearranged

#### 3D case: Gather inputs and get WTD

- took 550s to process the cell below

In [ ]:
# subsurface
start = time.time()
#visfile_subsurface = xdmf.VisFile(model_dir, domain=None, load_mesh=True, columnar=True, ats_version=1.4, model_time_unit='d')
visfile_subsurface = xdmf.VisFile(directory=model_dir,
                                  filename="ats_vis_data.h5", 
                                  mesh_filename="ats_vis_mesh.h5")
visfile_subsurface.loadMesh(columnar=True)
end = time.time()
print(f"Time cost for subsurface VisFile: {end - start:.6f} seconds")

# surface
start = time.time()
#visfile_surface = xdmf.VisFile(model_dir, domain='surface', load_mesh=True, ats_version=1.4, model_time_unit='d')

end = time.time()
print(f"Time cost for surface VisFile: {end - start:.6f} seconds")
visfile_surface = xdmf.VisFile(directory=model_dir,
                               domain="surface", 
                               filename="ats_vis_surface_data.h5" , 
                               mesh_filename="ats_vis_surface_mesh.h5")
visfile_surface.loadMesh()
# subsurface pressure (time, xy-space and soil columns)
start = time.time()
pressure_subsurface = visfile_subsurface.getArray('pressure')
end = time.time()
print(f"Time cost for getting pressure_subsurface: {end - start:.6f} seconds")

# get the WTD (based on surface ATS ID)
start = time.time()
surface_subsurface_IDs, head_rearranged, wtdep_rearranged, pwdep_rearranged = get_ats_wtd_pressurebased(pressure_subsurface=pressure_subsurface,
                                                                            visfile_surface=visfile_surface, visfile_subsurface= visfile_subsurface)
end = time.time()
print(f"Time cost for get_ats_wtd_pressurebased: {end - start:.6f} seconds")

In [ ]:
wtdep_rearranged

In [ ]:
wtdep_rearranged.shape # first dim is time, second dim is space

In [ ]:
surface_subsurface_IDs

#### Yi's operation to extract head at the starting and end points of a 2D transect.

In [ ]:
surface_x_coord = visfile_surface.centroids[:,0]
surface_y_coord = visfile_surface.centroids[:,1]

In [ ]:
print(surface_x_coord)
print(surface_x_coord.shape)
print(surface_y_coord)
print(surface_y_coord.shape)

In [ ]:
#[input] from 1-full_workflow_OakCreek.ipynb
start_coords, end_coords = (-1511015.74507677, 640547.2830956738), (-1511677.2648839361, 640389.829786001)

In [ ]:
# Define N
N = 7  # Number of closest points you want

# Compute Euclidean distances
dist_start = np.sqrt((surface_x_coord - start_coords[0])**2 + (surface_y_coord - start_coords[1])**2)
dist_end = np.sqrt((surface_x_coord - end_coords[0])**2 + (surface_y_coord - end_coords[1])**2)

# Get the indices of the N closest points
start_indices = np.argpartition(dist_start, N)[:N]
end_indices = np.argpartition(dist_end, N)[:N]

# Sort them by distance (optional, for cleaner output)
start_indices = start_indices[np.argsort(dist_start[start_indices])]
end_indices = end_indices[np.argsort(dist_end[end_indices])]

# Get the corresponding distances
start_distances = dist_start[start_indices]
end_distances = dist_end[end_indices]

# Print results
print(f"Top {N} closest points to start_coords:")
for i, (idx, dist) in enumerate(zip(start_indices, start_distances), 1):
    print(f"  {i}. Index: {idx}, Distance: {dist:.4f}")

print(f"\nTop {N} closest points to end_coords:")
for i, (idx, dist) in enumerate(zip(end_indices, end_distances), 1):
    print(f"  {i}. Index: {idx}, Distance: {dist:.4f}")

# To use just the closest point (like before)
start_index = start_indices[0]
end_index = end_indices[0]

In [ ]:
# extract time from model_dir/water_balance_170300020307.csv
# from xml, I expect times=11224 to 16059
# 11224 = 274+365*30 --> 2010.10.1?
# 16059 = 364+365*43 --> 2023.12.31?
time = visfile_surface.times

In [ ]:
print(time)

In [ ]:
output_hdf5_file = os.path.join(".", "bc_startend_raw.h5")

# Extract data based on computed indices
startpt_head  = head_rearranged[:, start_indices[0]]
endpt_head    = head_rearranged[:, end_indices[0]]
startpt_pwdep = pwdep_rearranged[:, start_indices[0]]
endpt_pwdep   = pwdep_rearranged[:, end_indices[0]]

# Write to HDF5
with h5py.File(output_hdf5_file, "w") as hdf:
    hdf.create_dataset("Time", data=time)
    hdf.create_dataset("startpt_head", data=startpt_head)
    hdf.create_dataset("endpt_head", data=endpt_head)
    hdf.create_dataset("startpt_pwdep", data=startpt_pwdep)
    hdf.create_dataset("endpt_pwdep", data=endpt_pwdep)

print(f"Data successfully written to {output_hdf5_file}")

### Double check the ponded water depth obtained

In [ ]:
import matplotlib.collections as mc
# Get mesh info directly
etype, coords, conn = xdmf.meshXYZ(model_dir, "ats_vis_surface_mesh.h5")

# Create polygon coordinates for triangles (use x, y coordinates only)
polygon_coords = [coords[c][:, :2] for c in conn]

# Create polygon collection
polygons = mc.PolyCollection(polygon_coords, edgecolor='k', cmap='Blues', linewidth=0.5)

# Get some data to visualize (e.g., pressure at first cycle)
# print("Available variables:")
# print(list(visfile_surface.d.keys()))
data = visfile_surface.get('surface-ponded_depth', visfile_surface.cycles[0])
polygons.set_array(data)
polygons.set_clim(vmin=0, vmax=0.01)  # Set specific range

# Define threshold
threshold = 1e-6
# Filter data
data_filtered = data[data > threshold]
print(f"Original data: {len(data)} elements")
print(f"Filtered data (>{threshold}): {len(data_filtered)} elements ({100*len(data_filtered)/len(data):.1f}%)")
print(f"\nFiltered statistics:")
print(f"  Min:     {data_filtered.min():.4g}")
print(f"  Max:     {data_filtered.max():.4g}")
print(f"  Mean:    {data_filtered.mean():.4g}")
print(f"  Median:  {np.median(data_filtered):.4g}")
print(f"  Std Dev: {data_filtered.std():.4g}")
print(f"  5th %:   {np.percentile(data_filtered, 5):.4g}")
print(f"  95th %:  {np.percentile(data_filtered, 95):.4g}")

print(f"\nOriginal (unfiltered) statistics:")
print(f"  Min:     {data.min():.4g}")
print(f"  Max:     {data.max():.4g}")
print(f"  Zeros:   {np.sum(data == 0)} ({100*np.sum(data == 0)/len(data):.1f}%)")
print(f"  <{threshold}: {np.sum(data < threshold)} ({100*np.sum(data < threshold)/len(data):.1f}%)")


# Plot
fig, ax = plt.subplots(figsize=(12, 5))
ax.add_collection(polygons)

# Zoom to region around start/end coordinates with buffer
# start_coords = (-1511015.74507677, 640547.2830956738)
# end_coords = (-1511677.2648839361, 640389.829786001)
# Calculate bounds with buffer (e.g., 10% extra on each side)
x_coords = [start_coords[0], end_coords[0]]
y_coords = [start_coords[1], end_coords[1]]
x_range = max(x_coords) - min(x_coords)
y_range = max(y_coords) - min(y_coords)
buffer = 0.1  # 10% buffer
ax.set_xlim(min(x_coords) - 1.5 * x_range, max(x_coords) + 0.4 * x_range)
ax.set_ylim(min(y_coords) - 4 * y_range, max(y_coords) + 4 * y_range)

ax.set_aspect('equal')
ax.set_xlabel('X [m]')
ax.set_ylabel('Y [m]')
plt.colorbar(polygons, ax=ax, label='surface-ponded_depth')

plt.title(f'Surface Mesh - Cycle {visfile_surface.cycles[0]}')

ax.plot(*start_coords, 'rx', markersize=10, label='Start')
ax.plot(*end_coords, 'ro', markersize=10, label='End')

plt.tight_layout()
plt.show()

In [ ]:
# Get mesh info directly
etype, coords, conn = xdmf.meshXYZ(model_dir, "ats_vis_surface_mesh.h5")

# Create polygon coordinates for triangles (use x, y coordinates only)
polygon_coords = [coords[c][:, :2] for c in conn]

# Create polygon collection
polygons = mc.PolyCollection(polygon_coords, edgecolor='k', cmap='Blues', linewidth=0.5)

# Get some data to visualize (e.g., pressure at first cycle)
# print("Available variables:")
# print(list(visfile_surface.d.keys()))
data = visfile_surface.get('surface-ponded_depth', visfile_surface.cycles[0])
polygons.set_array(data)
polygons.set_clim(vmin=0, vmax=0.01)  # Set specific range


# Plot
fig, ax = plt.subplots(figsize=(12, 5))
ax.add_collection(polygons)

# Highlight the polygon of the closest point to end_coords
closest_idx = end_indices[0]
highlighted_polygon = mc.PolyCollection([polygon_coords[closest_idx]], 
                                       edgecolor='red', 
                                       facecolors='none', 
                                       linewidth=3)
ax.add_collection(highlighted_polygon)

# Zoom to region around start/end coordinates with buffer
x_coords = [start_coords[0], end_coords[0]]
y_coords = [start_coords[1], end_coords[1]]
x_range = max(x_coords) - min(x_coords)
y_range = max(y_coords) - min(y_coords)
buffer = 0.1  # 10% buffer
ax.set_xlim(min(x_coords) - 0.4 * x_range, max(x_coords) - 0.3 * x_range)
ax.set_ylim(min(y_coords) - 1 * y_range, max(y_coords) + 1 * y_range)

ax.set_aspect('equal')
ax.set_xlabel('X [m]')
ax.set_ylabel('Y [m]')
plt.colorbar(polygons, ax=ax, label='surface-ponded_depth')

plt.title(f'Surface Mesh - Cycle {visfile_surface.cycles[0]}')

ax.plot(*start_coords, 'rx', markersize=10, label='Start')
ax.plot(*end_coords, 'ro', markersize=10, label='End')
ax.plot(surface_x_coord[closest_idx], surface_y_coord[closest_idx], 
        'go', markersize=8, label='Closest centroid to End')

# Plot top N closest points to end_coords
for i, idx in enumerate(end_indices):
    x, y = surface_x_coord[idx], surface_y_coord[idx]
    if i == 0:
        ax.plot(x, y, 'yo', markersize=8, label=f'Top {N} closest to End')
    else:
        ax.plot(x, y, 'yo', markersize=8)

    # Add rank number as text annotation
    ax.text(x, y, str(i+1), fontsize=10, ha='center', va='center', 
            color='red', fontweight='bold',
            bbox=dict(boxstyle='circle,pad=0.3', facecolor='white', edgecolor='red', alpha=0.8))

plt.tight_layout()
plt.show()

In [ ]:
# Extract head data for top N closest points to end_coords
endpt_heads = [head_rearranged[:, idx] for idx in end_indices]

# Plot
fig, ax = plt.subplots(figsize=(12, 6))

# Define colors for each line
colors = plt.cm.tab10(np.linspace(0, 1, N))

# Plot top 1 with solid line, rest with dashed lines
for i, (idx, head_data) in enumerate(zip(end_indices, endpt_heads)):
    if i == 0:
        # Top 1: solid line
        ax.plot(time, head_data, color=colors[i], linewidth=2, 
                label=f'Closest (idx={idx}, dist={end_distances[i]:.2f}m)')
    else:
        # Others: dashed lines
        ax.plot(time, head_data, color=colors[i], linewidth=1.5, linestyle='--',
                label=f'#{i+1} (idx={idx}, dist={end_distances[i]:.2f}m)')

ax.set_xlabel('Time [days]')
ax.set_ylabel('Head [m]')
ax.set_title(f'Head time series for top {N} closest points to end_coords')
ax.legend(loc='best', fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Extract water table depth data for top N closest points to end_coords
endpt_wtdeps = [wtdep_rearranged[:, idx] for idx in end_indices]

# Plot
fig, ax = plt.subplots(figsize=(12, 6))

# Define colors for each line
colors = plt.cm.tab10(np.linspace(0, 1, N))

# Plot top 1 with solid line, rest with dashed lines
for i, (idx, wtdep_data) in enumerate(zip(end_indices, endpt_wtdeps)):
    if i == 0:
        # Top 1: solid line
        ax.plot(time, -wtdep_data, color=colors[i], linewidth=2, 
                label=f'Closest (idx={idx}, dist={end_distances[i]:.2f}m)')
    else:
        # Others: dashed lines
        ax.plot(time, -wtdep_data, color=colors[i], linewidth=1.5, linestyle='--',
                label=f'#{i+1} (idx={idx}, dist={end_distances[i]:.2f}m)')

ax.set_xlabel('Time [days]')
ax.set_ylabel('Head [m]')
ax.set_title(f'Water table depth time series for top {N} closest points to end_coords')
ax.legend(loc='best', fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Extract ponded water depth data for top N closest points to end_coords
endpt_pwdeps = [pwdep_rearranged[:, idx] for idx in end_indices]

# Plot
fig, ax = plt.subplots(figsize=(12, 6))

# Define colors for each line
colors = plt.cm.tab10(np.linspace(0, 1, N))

# Plot top 1 with solid line, rest with dashed lines
for i, (idx, pwdep_data) in enumerate(zip(end_indices, endpt_pwdeps)):
    if i == 0:
        # Top 1: solid line
        ax.plot(time, pwdep_data, color=colors[i], linewidth=2, 
                label=f'Closest (idx={idx}, dist={end_distances[i]:.2f}m)')
    else:
        # Others: dashed lines
        ax.plot(time, pwdep_data, color=colors[i], linewidth=1.5, linestyle='--',
                label=f'#{i+1} (idx={idx}, dist={end_distances[i]:.2f}m)')

ax.set_xlabel('Time [days]')
ax.set_ylabel('Head [m]')
ax.set_title(f'Ponded water depth time series for top {N} closest points to end_coords')
ax.legend(loc='best', fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Plot pwdep_data from 2nd to 7th closest vs 1st closest
fig, ax = plt.subplots(figsize=(8, 6))

# Get data for the closest point (reference)
closest_pwdep = endpt_pwdeps[0]

# Define colors for each line
colors = plt.cm.tab10(np.linspace(0, 1, N))

# Plot 2nd to 7th closest against 1st closest
for i in range(1, N):
    ax.scatter(closest_pwdep, endpt_pwdeps[i], color=colors[i], alpha=0.5, s=10,
              label=f'#{i+1} (idx={end_indices[i]}, dist={end_distances[i]:.2f}m)')

# Add 1:1 reference line
min_val = min(closest_pwdep.min(), min([ep.min() for ep in endpt_pwdeps[1:]]))
max_val = max(closest_pwdep.max(), max([ep.max() for ep in endpt_pwdeps[1:]]))
ax.plot([min_val, max_val], [min_val, max_val], 'k--', linewidth=1, label='1:1 line')

ax.set_xlabel(f'Closest point ponded depth [m] (idx={end_indices[0]})')
ax.set_ylabel('Other points ponded depth [m]')
ax.set_title(f'Ponded water depth: 2nd-7th closest vs 1st closest to end_coords')
ax.legend(loc='best', fontsize=9)
ax.grid(True, alpha=0.3)
ax.set_aspect('equal')
plt.tight_layout()
plt.show()

In [ ]:
# Extract surface-vis-file based ponded water depth
# compare with the subsurface-vis-file based ponded water depth

# print("Available variables:")
# print(list(visfile_surface.d.keys()))
surface_ponded_depth = visfile_surface.getArray('surface-ponded_depth')
closest_pwdep_surface_vis = surface_ponded_depth[:, end_indices[0]]


# Plot pwdep_data from 2nd to 7th closest vs 1st closest
fig, ax = plt.subplots(figsize=(8, 6))

# Get data for the closest point (reference)
closest_pwdep = endpt_pwdeps[0]

ax.scatter(closest_pwdep, closest_pwdep_surface_vis, color='blue', alpha=0.5, s=10)

# Add 1:1 reference line
min_val = min(closest_pwdep.min(), min([ep.min() for ep in endpt_pwdeps[1:]]))
max_val = max(closest_pwdep.max(), max([ep.max() for ep in endpt_pwdeps[1:]]))
ax.plot([min_val, max_val], [min_val, max_val], 'k--', linewidth=1, label='1:1 line')

ax.set_xlabel(f'Ponded water depth based on subsurface pressure')
ax.set_ylabel('Other points ponded depth [m]')
ax.set_title(f'Ponded water depth from surface vis file')
#ax.legend(loc='best', fontsize=9)
ax.grid(True, alpha=0.3)
ax.set_aspect('equal')
plt.tight_layout()
plt.show()

In [ ]:
np.min(closest_pwdep_surface_vis)

### Yi's debug log to understand the get_ats_wtd_presrebased() above

In [ ]:
print(pressure_subsurface.shape)
print(pressure_subsurface[0,0,:])

In [ ]:
iz_coord = visfile_subsurface.centroids[:,:,-1]
#print(iz_coord)
print(iz_coord.shape)

In [ ]:
surface_centroids_rounded = np.round(visfile_surface.centroids[:, :2], 4)
print(visfile_surface.centroids.shape)
print(surface_centroids_rounded.shape)
surface_centroids_expanded = surface_centroids_rounded[:, np.newaxis, :]
print(surface_centroids_expanded.shape)

subsurface_centroids_rounded = np.round(visfile_subsurface.centroids[:, 0, :2], 4)
print(visfile_subsurface.centroids.shape)
print(subsurface_centroids_rounded.shape)
subsurface_centroids_expanded = subsurface_centroids_rounded[np.newaxis, :, :]
print(subsurface_centroids_expanded.shape)

In [ ]:
matches = np.all(surface_centroids_expanded == subsurface_centroids_expanded, axis=-1)
print(matches.shape)

In [ ]:
surface_indices, subsurface_indices = np.nonzero(matches)
surface_subsurface_IDs = np.column_stack((surface_indices, subsurface_indices))

In [ ]:
### Estimate WTD based on pressure
# pressure head
ih = (pressure_subsurface - patm) / (rho * g)
mask = ih > 0
first_false_idx = (~mask).argmax(axis=-1)
max_len = mask.shape[-1]
indices = np.arange(max_len)
mask2 = indices < first_false_idx[..., np.newaxis]
all_true_rows = (first_false_idx == 0)
mask2[all_true_rows] = True
sat_idx = mask2[:, :, ::-1].argmax(axis=-1)
sat_idx = mask2.shape[-1] - 1 - sat_idx
sat_idx[~mask2.any(axis=-1)] = 0
# WTD elevation
iH_rev = ih[np.arange(ih.shape[0])[:, None], np.arange(ih.shape[1]), sat_idx] + iz_coord[np.arange(ih.shape[1]), sat_idx] 
first_depth_to_centroid = visfile_surface.centroids[surface_subsurface_IDs[0][0], -1] - visfile_subsurface.centroids[surface_subsurface_IDs[0][1], -1, -1]
# WTD (from surface)
head_pressure_based = -(iz_coord[:,-1]+first_depth_to_centroid - iH_rev) #[added by Yi]
wtd_pressure_based = np.maximum(iz_coord[:,-1]+first_depth_to_centroid - iH_rev, 0) 
assert pressure_subsurface[:,:,1].shape == wtd_pressure_based.shape, f"Shape mismatch: Error in WTD calculation"

### Re-arrange WTD in accordance to surface IDs
# As pressure obtained from subsurface does not follow surface ID, re-arrange this:
wtd_pressure_rearranged = wtd_pressure_based[:, subsurface_indices]
head_rearranged = head_pressure_based[:, subsurface_indices]

In [ ]:
print(ih.shape)
print(mask.shape)
print(first_false_idx.shape)

In [ ]:
tmp_t, tmp_n = 1000, 1000
print(ih[tmp_t, tmp_n])
print(mask[tmp_t,tmp_n,:])
print(first_false_idx[tmp_t, tmp_n])
print(mask2[tmp_t,tmp_n])
print(sat_idx.shape)
print(sat_idx[tmp_t, tmp_n])

print(iH_rev[tmp_t, tmp_n])
print(ih[tmp_t, tmp_n][sat_idx[tmp_t, tmp_n]])
print(iz_coord[tmp_n][sat_idx[tmp_t, tmp_n]])
print(head_pressure_based[tmp_t, tmp_n])
print(wtd_pressure_based[tmp_t, tmp_n])

In [ ]:
#tmp_t, tmp_n = random.randint(0, 4836-1), random.randint(0, 24219-1)
#print([tmp_t, tmp_n])
tmp_t, tmp_n = 0, 0

print(ih[tmp_t, tmp_n])
print(mask[tmp_t,tmp_n,:])
print(first_false_idx[tmp_t, tmp_n])
print(mask2[tmp_t,tmp_n])
print(sat_idx.shape)
print(sat_idx[tmp_t, tmp_n])

print(iH_rev[tmp_t, tmp_n])
print(ih[tmp_t, tmp_n][sat_idx[tmp_t, tmp_n]])
print(iz_coord[tmp_n][sat_idx[tmp_t, tmp_n]])
print(head_pressure_based[tmp_t, tmp_n])
print(wtd_pressure_based[tmp_t, tmp_n])

In [ ]:
print(first_depth_to_centroid)

In [ ]:
print(surface_subsurface_IDs.shape)

In [ ]:
head_pressure_based.shape

In [ ]:
tmp_indices = np.where(head_pressure_based > 0)
print(tmp_indices)
print(tmp_indices[0].shape)
print(tmp_indices[0].shape[0]/(head_pressure_based.shape[0]*head_pressure_based.shape[1])) # percent of element with surface water

In [ ]:
max_index = np.unravel_index(np.argmax(head_pressure_based), head_pressure_based.shape)
print(max_index)
print(head_pressure_based[max_index])

#### 2D case: Gather inputs and get WTD

In [ ]:
# subsurface
start = time.time()
visfile_subsurface = xdmf.VisFile(directory=model_dir, prefix='vis_subsurface',domain=None, load_mesh=True, columnar=True, ats_version=1.4, model_time_unit='d')
end = time.time()
print(f"Time cost for subsurface VisFile: {end - start:.6f} seconds")

# surface
start = time.time()
visfile_surface = xdmf.VisFile(model_dir, domain='surface', prefix ='vis', load_mesh=True, ats_version=1.4, model_time_unit='d')
end = time.time()
print(f"Time cost for surface VisFile: {end - start:.6f} seconds")

# subsurface pressure (time, xy-space and soil columns)
start = time.time()
pressure_subsurface = visfile_subsurface.getArray('pressure')
end = time.time()
print(f"Time cost for getting pressure_subsurface: {end - start:.6f} seconds")

# get the WTD (based on surface ATS ID)
start = time.time()
wtd_pressure_rearranged, surface_subsurface_IDs = get_ats_wtd_pressurebased(pressure_subsurface=pressure_subsurface,
                                                                            visfile_surface=visfile_surface, visfile_subsurface= visfile_subsurface)
end = time.time()
print(f"Time cost for get_ats_wtd_pressurebased: {end - start:.6f} seconds")

In [ ]:
wtd_pressure_rearranged

In [ ]:
wtd_pressure_rearranged.shape